# Сравнение двух embedding-моделей
- **BGE** (`BAAI/bge-base-en-v1.5`) — через LangChain Chroma
- **PPLX** (`pplx-embed-v1-0.6b`) — через ChromaDB напрямую

In [1]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import chromadb
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from scripts.pplx_embed import PplxEmbedFunction
from dotenv import load_dotenv

load_dotenv()

/Users/sergey/Desktop/Moodle_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# === BGE коллекция (через LangChain) ===
BGE_CHROMA_DIR = os.environ["MOODLE_CHROMA_DB_DIR"]
BGE_COLLECTION = os.environ.get("MOODLE_COLLECTION_NAME", "moodle_docs")

bge_embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
bge_store = Chroma(
    collection_name=BGE_COLLECTION,
    embedding_function=bge_embeddings,
    persist_directory=BGE_CHROMA_DIR,
)
print(f"BGE collection: {bge_store._collection.count()} чанков")

/var/folders/h2/r9wh0x750xq4v5z33ylwrzw40000gn/T/ipykernel_14517/1919293459.py:5: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  bge_embeddings = HuggingFaceBgeEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1794.93it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BGE collection: 2697 чанков


In [3]:
# === PPLX коллекция (через chromadb напрямую) ===
PPLX_CHROMA_DIR = "/Users/sergey/Desktop/Moodle_RAG/scripts/Notebooks/chroma_db_pplx"
PPLX_COLLECTION = "moodle_docs_pplx"

pplx_embed = PplxEmbedFunction()
pplx_client = chromadb.PersistentClient(path=PPLX_CHROMA_DIR)
pplx_col = pplx_client.get_collection(
    name=PPLX_COLLECTION,
    embedding_function=pplx_embed,
)
print(f"PPLX collection: {pplx_col.count()} чанков")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1947.71it/s, Materializing param=norm.weight]                              


PPLX collection: 3395 чанков


In [4]:
# === Ground truth ===
# Формат: (запрос, список подстрок из ожидаемых source-файлов)
# Если хотя бы один результат содержит подстроку — считаем hit.

test_queries = [
    ("How to create a new course in Moodle?",
     ["Adding_a_new_course"]),

    ("How to set up gradebook?",
     ["Gradebook", "Grades"]),

    ("How to add students to a course?",
     ["Add_students", "Course_enrolment"]),

    ("How to create a quiz?",
     ["Quiz"]),

    ("How to upload files in Moodle?",
     ["File", "Upload"]),

    ("How to set up self enrolment?",
     ["Self_enrolment", "enrolment"]),

    ("How to backup a course?",
     ["Backup", "backup"]),

    ("How to assign roles?",
     ["role", "Role"]),

    ("How to use forums in Moodle?",
     ["Forum", "forum"]),

    ("How to configure authentication?",
     ["auth", "Auth", "authentication"]),
]

In [5]:
K = 5  # top-K результатов

def search_bge(query: str, k: int = K) -> list[dict]:
    """Возвращает список метаданных из BGE коллекции."""
    results = bge_store.similarity_search_with_score(query, k=k)
    return [
        {"source": doc.metadata.get("doc_title", "") + " " + doc.metadata.get("source_file", ""),
         "score": 1 - score,
         "text": doc.page_content[:120]}
        for doc, score in results
    ]

def search_pplx(query: str, k: int = K) -> list[dict]:
    """Возвращает список метаданных из PPLX коллекции."""
    results = pplx_col.query(query_texts=[query], n_results=k)
    out = []
    for meta, dist, doc in zip(
        results["metadatas"][0],
        results["distances"][0],
        results["documents"][0],
    ):
        out.append({
            "source": meta.get("source", ""),
            "score": -dist,  # меньше расстояние = лучше
            "text": doc[:120],
        })
    return out

In [6]:
def is_hit(results: list[dict], expected: list[str]) -> bool:
    """Есть ли хотя бы один результат, содержащий одну из expected подстрок."""
    for r in results:
        for exp in expected:
            if exp.lower() in r["source"].lower():
                return True
    return False

def reciprocal_rank(results: list[dict], expected: list[str]) -> float:
    """1/позиция первого релевантного результата (0 если нет)."""
    for i, r in enumerate(results):
        for exp in expected:
            if exp.lower() in r["source"].lower():
                return 1.0 / (i + 1)
    return 0.0

In [7]:
bge_hits, pplx_hits = 0, 0
bge_rr, pplx_rr = 0.0, 0.0

print(f"{'Запрос':<45} {'BGE':>10} {'PPLX':>10}  {'BGE RR':>8} {'PPLX RR':>8}")
print("=" * 95)

for query, expected in test_queries:
    r_bge = search_bge(query)
    r_pplx = search_pplx(query)

    h_bge = is_hit(r_bge, expected)
    h_pplx = is_hit(r_pplx, expected)
    rr_bge = reciprocal_rank(r_bge, expected)
    rr_pplx = reciprocal_rank(r_pplx, expected)

    bge_hits += h_bge
    pplx_hits += h_pplx
    bge_rr += rr_bge
    pplx_rr += rr_pplx

    q_short = query[:43]
    print(f"{q_short:<45} {'✅' if h_bge else '❌':>10} {'✅' if h_pplx else '❌':>10}  {rr_bge:>8.3f} {rr_pplx:>8.3f}")

n = len(test_queries)
print("=" * 95)
print(f"{'Hit Rate @' + str(K):<45} {bge_hits/n:>10.1%} {pplx_hits/n:>10.1%}")
print(f"{'MRR @' + str(K):<45} {bge_rr/n:>10.3f} {pplx_rr/n:>10.3f}")

Запрос                                               BGE       PPLX    BGE RR  PPLX RR


/Users/sergey/.cache/huggingface/modules/transformers_modules/perplexity_hyphen_ai/pplx_hyphen_embed_hyphen_v1_hyphen_0_dot_6b/1dc2ea99a948a2f17b103949ad02b0194a20c0a8/modeling.py:62: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  "full_attention": create_causal_mask(


How to create a new course in Moodle?                  ✅          ✅     1.000    1.000
How to set up gradebook?                               ❌          ✅     0.000    1.000
How to add students to a course?                       ✅          ✅     1.000    1.000
How to create a quiz?                                  ✅          ✅     1.000    1.000
How to upload files in Moodle?                         ✅          ✅     1.000    1.000
How to set up self enrolment?                          ✅          ✅     1.000    1.000
How to backup a course?                                ✅          ✅     1.000    1.000
How to assign roles?                                   ✅          ✅     0.500    1.000
How to use forums in Moodle?                           ✅          ✅     1.000    1.000
How to configure authentication?                       ✅          ✅     1.000    1.000
Hit Rate @5                                        90.0%     100.0%
MRR @5                                             0.850      

In [8]:
# === Детальное сравнение по одному запросу ===
demo_query = "How to create a new course in Moodle?"

print(f"🔍 Запрос: {demo_query}\n")

print("--- BGE top-5 ---")
for i, r in enumerate(search_bge(demo_query), 1):
    print(f"  #{i} | sim={r['score']:.4f} | {r['source']}")
    print(f"       {r['text']}")

print("\n--- PPLX top-5 ---")
for i, r in enumerate(search_pplx(demo_query), 1):
    print(f"  #{i} | dist={-r['score']:.4f} | {r['source']}")
    print(f"       {r['text']}")

🔍 Запрос: How to create a new course in Moodle?

--- BGE top-5 ---
  #1 | sim=0.7390 | Adding a new course /Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md/403__en__Adding_a_new_course/chunk_0000.md
       # Adding a new course  
## Adding a course
By default a regular teacher can't add a new course. To add a new course to M
  #2 | sim=0.6042 | Installing Moodle /Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md/403__en__Installing_Moodle/chunk_0017.md
       ### Installation is complete :)
* Create a new course: You can now access Moodle through your web browser (using the sam
  #3 | sim=0.5983 | Managing a Moodle course /Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md/403__en__Managing_a_Moodle_course/chunk_0000.md
       # Managing a Moodle course  
A _course_ in Moodle is an area where a teacher will add resources and activities for their
  #4 | sim=0.5977 | Admin quick guide /Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md/403__en__Admin_q

In [9]:
# === Детальное сравнение по одному запросу ===
demo_query = "Как создать новый курс на Moodle?"

print(f"🔍 Запрос: {demo_query}\n")

print("--- BGE top-5 ---")
for i, r in enumerate(search_bge(demo_query), 1):
    print(f"  #{i} | sim={r['score']:.4f} | {r['source']}")
    print(f"       {r['text']}")

print("\n--- PPLX top-5 ---")
for i, r in enumerate(search_pplx(demo_query), 1):
    print(f"  #{i} | dist={-r['score']:.4f} | {r['source']}")
    print(f"       {r['text']}")

🔍 Запрос: Как создать новый курс на Moodle?

--- BGE top-5 ---
  #1 | sim=0.3515 | Features /Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md/403__en__Features/chunk_0000.md
       # Features  
Moodle is a free, online Learning Management system enabling educators to create their own private website 
  #2 | sim=0.3248 | About Moodle /Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md/403__en__About_Moodle/chunk_0003.md
       ### Teach and learn in the way you prefer
With over 20 years of development guided by social constructionist pedagogy, M
  #3 | sim=0.3132 | Moodle key terms /Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md/403__en__Moodle_key_terms/chunk_0001.md
       ## Activities
* An Activity in Moodle is a feature where students learn by interacting with each other or with their tea
  #4 | sim=0.3030 | Standards /Users/sergey/Desktop/Moodle_RAG/data/moodle_docs/chunks_md/403__en__Standards/chunk_0000.md
       # Standards  
Moodle is a global lea